# 08 — Climate Seasonality Stratification & Hurdle Stage 1 Diagnostics

Phân tầng sai số theo Mùa Khí hậu (Mùa Mưa T5-T10 vs Mùa Khô T11-T4 theo Target Date t+h) và đánh giá chất lượng phân loại nhị phân Stage 1 Hurdle (ROC-AUC, Brier score, PR-AUC).

### Project Guardrails & Execution Baseline:
- **Protocol Căn cứ:** `training_forecasting_evaluation_protocol_v1_0_21.md` và `spec_module_result_analysis_v2.md`
- **Zero Final Test Access Rule:** Chỉ nạp dữ liệu Validation (724 origins, 2018-05-01 đến 2020-04-23, H=7).
- **Purged Metrics Integrity:** Tuyệt đối không tính hoặc hiển thị CR-MAE@7, CR-Bias@7, RainDay-MAE, hay Bias.
- **Descriptive Cost Principle:** Số liệu tài nguyên chỉ dùng mô tả (PROP-07), không dùng để xếp hạng mô hình.


In [1]:
# Bootstrap root directory
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / 'src').exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


In [2]:
from src.analysis.loaders import load_all_forecast_matrices, load_hurdle_probabilities, load_hurdle_threshold_metadata
from src.analysis.aggregate import compute_seasonal_breakdown
from src.analysis.hurdle_diagnostics import compute_all_hurdle_stage1_diagnostics

forecasts, y_true, origin_dates = load_all_forecast_matrices()
prob_dict = load_hurdle_probabilities()
thresh_meta = load_hurdle_threshold_metadata()

# 1. Seasonal breakdown
print("1. CLIMATE SEASONAL BREAKDOWN (Target Date t+h):")
df_seasonal = compute_seasonal_breakdown(forecasts, y_true, origin_dates)
display(df_seasonal.head(15))

# 2. Hurdle Stage 1 classification diagnostics
print("\n2. HURDLE STAGE 1 CLASSIFICATION DIAGNOSTICS (WMO RR1 >= 1.0 mm):")
df_diag = compute_all_hurdle_stage1_diagnostics(prob_dict, thresh_meta)
display(df_diag)


1. CLIMATE SEASONAL BREAKDOWN (Target Date t+h):


,scenario_id,overall_MAE,overall_RMSE,rainy_MAE,rainy_RMSE,dry_MAE,dry_RMSE,delta_seasonal_MAE,n_rainy_points,n_dry_points
0,SC-30,3.148493,6.687314,5.169789,8.761083,1.104737,3.511096,4.065052,2548,2520
1,SC-34,3.156542,6.631751,5.156972,8.679038,1.133886,3.505178,4.023085,2548,2520
2,SC-29,3.158415,6.668934,5.174489,8.738809,1.119940,3.496890,4.054549,2548,2520
3,SC-35,3.168312,6.643506,5.179411,8.698792,1.134868,3.500425,4.044543,2548,2520
4,SC-33,3.169547,6.642366,5.188940,8.697982,1.127717,3.498111,4.061224,2548,2520
5,SC-31,3.193921,6.713059,5.234549,8.796922,1.130620,3.519287,4.103929,2548,2520
6,SC-32,3.233099,6.705509,5.281959,8.780430,1.161474,3.531955,4.120485,2548,2520
7,SC-28,3.237691,6.742826,5.306792,8.834956,1.145601,3.537352,4.161191,2548,2520
8,SC-42,3.318662,6.690733,5.314684,8.775148,1.300462,3.488610,4.014222,2548,2520
9,SC-44,3.334091,6.709963,5.323827,8.816025,1.322246,3.458553,4.001581,2548,2520



2. HURDLE STAGE 1 CLASSIFICATION DIAGNOSTICS (WMO RR1 >= 1.0 mm):


,hurdle_family,feature_set,optimal_tau,ROC_AUC,Brier_Score,PR_AUC,Precision,Recall,F1_Score,Specificity,TP,FP,FN,TN
0,xgboost,F2,0.70,0.881997,0.134831,0.833959,0.828420,0.594332,0.692118,0.892382,1405,291,959,2413
1,random_forest,F2,0.70,0.877426,0.139269,0.830637,0.830621,0.582910,0.685061,0.896080,1378,281,986,2423
2,xgboost,F1,0.70,0.877304,0.136320,0.822377,0.825070,0.626481,0.712190,0.883876,1481,314,883,2390
3,random_forest,F1,0.70,0.877238,0.137596,0.824460,0.827724,0.613790,0.704882,0.888314,1451,302,913,2402
4,logistic_linear,F2,0.65,0.877080,0.135172,0.818231,0.826409,0.706853,0.761970,0.870192,1671,351,693,2353
5,logistic_linear,F1,0.65,0.876223,0.136148,0.814033,0.818957,0.690778,0.749426,0.866494,1633,361,731,2343
6,xgboost,F3,0.70,0.875793,0.137520,0.818737,0.824582,0.584602,0.684158,0.891272,1382,294,982,2410
7,lightgbm,F1,0.80,0.873822,0.138604,0.818478,0.842410,0.307530,0.450573,0.949704,727,136,1637,2568
8,random_forest,F3,0.70,0.873572,0.140310,0.822666,0.829122,0.582910,0.684550,0.894970,1378,284,986,2420
9,lightgbm,F2,0.80,0.872344,0.139038,0.817504,0.841388,0.287225,0.428256,0.952663,679,128,1685,2576
